# PROC CHART로 지역 전력망 부하와 정전 프로파일링하기

## 요약

전력망 운영 분석가가 PROC CHART를 사용해 네 개 서비스 지역과 네 개 발전원에 걸친 급전 회로 계량기 판독값의 합성 표본을 프로파일링합니다. 이 노트북은 세로 막대, 가로 막대, 블록, 원형 차트를 차례로 살펴보면서 발전 믹스를 요약하고, 지역별 평균 및 총 부하를 비교하고, 시간대별 저녁 수요 피크를 드러내고, 발전원별 정전 시간을 순위화합니다. 에너지·유틸리티 팀이 그래픽 보고서에 착수하기 전에 실행하는 빠른 텍스트 출력 탐색이 바로 이런 종류입니다. DATA 스텝은 1,200개 행을 요청하지만, 이 노트북은 비라이선스 모드에서 렌더링되어 작업 데이터셋이 처음 100개 판독값으로 제한되므로, 아래의 모든 차트는 그 100개 행 표본을 요약합니다.

## 데이터 소스

| 데이터셋 | 행 수 | 설명 |
|---------|------|-------------|
| `grid_ops` | 100 (합성 표본) | 지역 전력망의 급전 회로 계량기 판독값 한 건당 한 행으로, `call streaminit(20260531)`과 `rand()`를 사용해 인라인으로 생성됩니다. DATA 스텝 루프는 1,200개 판독값을 요청하지만, 비라이선스 모드가 저장 데이터셋을 100개 관측값으로 제한하므로 아래 차트들은 그 100개 행을 설명합니다. |

# PROC CHART로 지역 전력망 부하와 정전 프로파일링하기

PROC CHART는 리스팅에 문자 기반의 막대, 블록, 원형 차트를 직접 생성합니다 — ODS Graphics 장치가 필요 없습니다. 그래서 잘 다듬어진 GCHART나 SGPLOT 시각화를 구축하기 전에 부하 및 신뢰성 데이터의 형태를 *보고* 싶어 하는 전력망 운영 팀에게 빠른 1차 탐색 도구가 됩니다.

이 노트북에서 우리는 다음을 수행합니다.

1. 지역 전력망 운영자를 위한 급전 회로 계량기 판독값의 합성된 한 달치 데이터를 생성합니다.
2. **발전 믹스**(발전원별 판독값 비중)를 차트로 그립니다.
3. 서비스 지역 간 **평균 및 총 공급 부하**를 비교합니다.
4. 시간대별 **저녁 수요 피크**를 드러냅니다.
5. **블록 차트**를 사용해 지역과 발전원을 교차합니다.
6. 발전원별 **정전 시간**을 순위화하여 신뢰성이 가장 낮은 급전을 찾습니다.

아래의 모든 문과 옵션은 표준 SAS 9.4 PROC CHART 구문입니다.

## 1단계 — 합성 운영 데이터 생성

아래 DATA 스텝은 1,200회 반복 루프로 계량기 판독값을 만들어냅니다. 각 판독값에는 서비스 지역, 발전원(가스가 지배적이고 풍력, 태양광, 수력이 나머지를 구성), 시간대가 할당됩니다. 부하는 17:00–21:00 저녁 구간에서 더 높으며(이 판독값들을 피크로 표시함), 정전 시간은 포아송 분포에서 추출합니다. `call streaminit`이 난수 시드를 고정하여 데이터를 재현 가능하게 만듭니다.

로그의 NOTE는 실질적인 결과를 보여줍니다. 이 실행은 비라이선스 모드이므로 저장된 `grid_ops` 데이터셋이 그 판독값들 중 처음 100개(100개 행, 7개 열)로 제한됩니다. 이어지는 모든 차트는 그 100개 행 표본을 요약하며, 각 PROC CHART 로그가 100개 행을 읽었음을 확인해 줍니다.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
데이터 grid_ops;
    호출 streaminit(20260531);
    길이 region $12 source $9 peak_flag $3;
    배열 regions[4] $12 _temporary_
        ('North','South','East','West');
    반복 meter_id = 1 까지 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        만약 u < 0.42 이면 source = 'Gas';
        아니면 만약 u < 0.70 이면 source = 'Wind';
        아니면 만약 u < 0.88 이면 source = 'Solar';
        아니면 source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        만약 hour >= 17 그리고 hour <= 21 이면 반복;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        종료;
        아니면 반복;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        종료;
        만약 load_mwh < 0 이면 load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        출력;
    종료;
    제거 r u BASE;
실행;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## 2단계 — 발전 믹스

각 발전원은 판독값의 몇 퍼센트를 차지할까요? `TYPE=PERCENT`를 사용한 세로 막대 차트가 이에 직접 답합니다. 막대 높이는 각 발전원 범주에 속하는 전체 관측값의 백분율입니다. `source`가 문자 변수이므로 PROC CHART는 그 값을 자동으로 이산 범주로 취급합니다.

In [2]:
처리 chart 데이터=grid_ops;
    VBAR source / type=PERCENT;
실행;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 3단계 — 지역별 평균 공급 부하

이제 개수 세기에서 측정값 요약으로 전환합니다. `SUMVAR=`에 `load_mwh`를 지정하고 `TYPE=MEAN`을 사용하면 막대 길이가 지역별 평균 부하가 되며, 통계 열을 명시적으로 요청합니다. `MEAN`은 각 막대 옆에 평균을 출력하고 `CFREQ`는 누적 빈도 열을 추가합니다. North가 가장 높은 평균 공급 부하(평균 약 28.0 MWh)를 나타내는데, 이는 데이터에 내장된 지역 오프셋과 일치하며, South가 가장 낮습니다(약 19.8 MWh).

또한 막대를 가장 낮은 것부터 가장 높은 평균 순으로 정렬하려는 의도로 `ASCENDING`을 전달합니다. 출력에서 막대가 **재정렬되지 않았음**에 유의하세요 — 막대는 범주 순서(West, South, East, North)로 나타나며, 평균 24.2, 19.8, 21.7, 28.0은 짧은 것부터 긴 것 순으로 이어지지 않습니다. 차트 통계로 막대를 재정렬하는 기능은 이 PROC CHART 구현에 아직 연결되어 있지 않으므로, `ASCENDING`/`DESCENDING`은 받아들여지지만 현재 아무 효과가 없습니다. 순위는 대신 출력된 `Mean` 열에서 읽으세요.

In [3]:
처리 chart 데이터=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
실행;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 4단계 — 지역별 총 부하

여기서 `TYPE=SUM`은 각 막대를 지역의 평균이 아니라 *총* 공급 부하로 만들므로, 더 높은 막대는 표본 판독값 전반에 걸쳐 가장 많은 총 에너지를 공급한 지역을 표시합니다. 여기서도 North가 총 공급 부하에서 선두입니다.

이 문은 또한 `SUBGROUP=peak_flag`를 요청하여, 판독값이 저녁 피크 구간에 속했는지 여부에 따라 각 막대를 분할하도록 PROC CHART에 요청합니다. SAS에서는 이것이 각 막대를 하위 그룹 수준별로 서로 다른 글리프로 세분하고 범례를 출력합니다. 이 구현은 아직 하위 그룹 세분화를 렌더링하지 않으므로(문서화된 기능 격차), 막대는 세그먼트별 분해 없이 단색 `*` 연속으로 나옵니다 — 막대 *합계*는 정확하지만, 피크 대 비피크 분할은 여기에 표시되지 않습니다. 피크 시간에 얼마나 많은 부하가 몰리는지 보려면 5단계의 시간대별 뷰를 사용하세요.

In [4]:
처리 chart 데이터=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
실행;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 5단계 — 하루 동안의 부하 형태

`hour`는 연속형이므로, 4시간 중심점(2, 6, 10, 14, 18, 22)에 명시적인 `MIDPOINTS=` 목록으로 구간화를 직접 고정합니다. 각 막대는 해당 시간 근처 판독값의 평균 공급 부하를 나타냅니다. 18을 중심으로 한 막대가 두드러져야 합니다 — 이것이 우리가 데이터에 내장한 저녁 수요 피크입니다.

In [5]:
처리 chart 데이터=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
실행;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## 6단계 — 지역별 발전원 (블록 차트)

BLOCK 차트는 작은 이원 표를 블록 격자로 렌더링합니다. `GROUP=source`와 `SUMVAR=load_mwh / TYPE=MEAN`을 사용하면, 차트가 각 지역을 그 지역에 공급하는 발전원과 교차하며, 블록 높이는 평균 부하에 비례합니다 — 어떤 지역/발전원 조합이 가장 무거운 평균 부하를 감당하는지 포착하는 간결한 방법입니다.

In [6]:
처리 chart 데이터=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
실행;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 7단계 — 원형 차트로 본 발전 믹스

2단계의 동일한 발전원 비중 정보를 원형으로 그린 것입니다. `TYPE=PERCENT`를 사용한 PIE는 각 조각을 전체 판독값 대비 백분율로 크기를 정하고, 그림 옆에 조각 백분율 범례를 출력합니다.

In [7]:
처리 chart 데이터=grid_ops;
    PIE source / type=PERCENT;
실행;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 8단계 — 발전원별 정전 시간

마지막으로 신뢰성을 순위화합니다. `SUMVAR=outage_min`과 `TYPE=SUM`은 발전원별 정전 시간을 합산합니다. 성능이 가장 나쁜 발전원을 맨 위로 띄우려고 `DESCENDING`을 전달하지만, 3단계에서처럼 막대는 재정렬되지 않습니다 — 막대는 범주 순서(Hydro, Wind, Gas, Solar)로 출력되며 막대 재정렬은 아직 구현되지 않았습니다. 순위는 출력된 `Sum` 열에서 읽으세요. 이 표본에서 Gas가 가장 많은 총 정전 시간(122)을 차지하며, Wind(64), Solar(43), Hydro(38)를 크게 앞섭니다. 이는 발전 믹스와 일치합니다 — Gas가 가장 많은 판독값을 담당하므로 전체적으로 가장 많은 정전 시간이 누적됩니다.

In [8]:
처리 chart 데이터=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum DESCENDING;
실행;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## 결과 해석

차트들을 함께 읽으면 운영 팀에게 빠른 상황 파악을 제공합니다(이 실행이 포착한 100개 행 표본 기준).

- **발전 믹스 (2단계와 7단계).** Gas가 가장 큰 판독값 비중(약 45%)을 차지하고, Wind가 2위(약 28%), Hydro와 Solar가 그 뒤를 따릅니다(약 14%와 13%) — 세로 막대와 원형이 같은 이야기를 두 가지 방식으로 전하며, 유용한 상호 검증이 됩니다.
- **지역별 부하 (3단계와 4단계).** 평균 부하 HBAR는 North가 가장 높은 평균 공급 부하(평균 약 28 MWh)를, South가 가장 낮은 값(약 20 MWh)을 보임을 나타내며, 이는 데이터에 내장된 지역 오프셋과 일치합니다. SUM 차트는 North가 또한 가장 많은 총 에너지를 공급함을 확인해 줍니다.
- **일별 부하 형태 (5단계).** 18시를 중심으로 한 중심점 막대가 확연히 가장 높아, 우리가 데이터에 내장한 17:00–21:00 수요 피크를 확인해 줍니다 — 유틸리티가 수요 반응과 용량 계획에 집중할 바로 그 지점입니다.
- **신뢰성 (8단계).** 발전원별 정전 시간을 합산하면 이 표본에서 Gas가 가장 큰 가동 중단 기여자(122분)로 드러나며, 정비 검토의 자연스러운 다음 대상이 됩니다 — 다만 이는 대부분 Gas가 가장 많은 판독값을 담당하는 데서 비롯됩니다.

여기서 사용한 두 가지 표시 옵션 — `ASCENDING`/`DESCENDING` 막대 재정렬(3단계와 8단계)과 `SUBGROUP=` 막대 세분화(4단계) — 은 파서가 받아들이지만 이 PROC CHART 구현이 아직 렌더링하지 않으므로, 순위와 분할은 막대 순서나 음영이 아니라 출력된 통계 열에서 읽습니다.

PROC CHART는 문자 출력 전용이므로, 이사회 발표에 적합한 시각화를 위해서라면 팀은 이 동일한 뷰들을 PROC GCHART나 PROC SGPLOT으로 옮길 것입니다. 하지만 새로운 추출에 대한 무설정 1차 탐색으로서, 이 차트들은 운영상의 질문 — 믹스, 규모, 시점 — 에 몇 초 만에 답합니다.